# Hidden CKD

#### Variables
- Date of event: Date of the screening
- Gender: Gender of the patient (M: Male, F: Female)
- Ethnicity: The ethnicity of the participant
- D.O.B.: The date of birth of the participant
- Age: Age of the patient (years)
- Height (cm): Height of the participant in cm
- Weight (kg): Weight of the participant in kg
- BMI: BMI of the participant
- BMI Category: Classification of the particpant BMI according to NICE guidelines
- Systolic, Diastolic: The systolic and diastolic of the partcipants
- BP Category: Classification of the particpant BP according to NICE guidelines
- Medical Conditions: Medical conditions the patient has (High blood pressure, Diabetes, Kidney disease, Heart disease and Other
- What medications/tablets are you currently taking?: The kinds of medication the participants are taking (Cholesterol, BP, Diabetes, Other)
- Name of blood pressure medication / Tablets
- Name of blood pressure medication / Tablets
- Name of blood pressure medication / Tablets
- Do you have a family history of kidney disease?: Whether or not the participant has a family history of kidney disease
- uACR: uACR level of the participant as measured using a urine dipstick (Normal, Abnormal, High Abnormal)

In [5]:
import numpy as np
import pandas as pd
from src.config import PROCESSED_DATA_DIR

In [6]:
filename = PROCESSED_DATA_DIR / 'hidden_ckd_processed.csv'
data = pd.read_csv(filename)
data.head()

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,Has_Diabetes,Has_KD,Has_HD,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk
0,23/10/2022,Male,Black Caribbean,Black,True,21/05/1946,76.5,>70,161.0,64.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
1,23/10/2022,Male,Black African (West Africa),Black,True,25/01/1970,52.8,41-55,163.0,78.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
2,26/08/2023,Male,Black Caribbean,Black,True,14/07/2005,18.1,<25,167.0,91.0,...,False,False,False,False,False,False,False,Definitely not,Normal,Low
3,28/04/2023,Male,Black Caribbean,Black,True,25/04/1969,54.0,41-55,168.0,87.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
4,06/11/2022,Female,Black African (West Africa),Black,True,03/11/1979,43.0,41-55,187.0,109.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate


# **Exploratory Data Analysis**

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [8]:
data.describe()

,Age,Height,Weight,BMI,Systolic,Diastolic,Pulse_Pressure
count,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000
mean,50.901457,165.813309,81.799761,29.841207,136.210198,82.668054,53.542144
std,14.165017,9.140738,15.422907,5.720421,20.212144,11.863947,15.003352
min,14.400000,111.500000,45.500000,19.000000,88.000000,45.000000,11.000000
25%,42.100000,159.000000,71.000000,25.800000,123.000000,75.000000,44.000000
50%,52.900000,165.000000,80.200000,29.200000,134.000000,82.000000,52.000000
75%,60.300000,171.000000,90.000000,32.900000,146.000000,90.000000,61.000000
max,92.100000,198.000000,169.000000,62.100000,243.000000,141.000000,154.000000


In [9]:
#pairwise Pearson correlations between numeric variables
cr = data[['Age', 'Height', 'Weight', 'BMI', 'Systolic', 'Diastolic']].corr(method='pearson')

fig = go.Figure(go.Heatmap(
    x=cr.columns,
    y=cr.columns,
    z=cr.values.tolist(),
    colorscale='rdylgn', zmin=-1, zmax=1
))

fig.show()

In [10]:
#creating a function to group variables

def group_stats(column_name):
    '''
    Groups the data and runs some basic stats
    '''
    table = {
        column_name: data[column_name].sort_values().unique(),
        'Mean Age': data.groupby(column_name)['Age'].mean().round(2),
        'Mean Height (cm)': data.groupby(column_name)['Height'].mean().round(2),
        'Mean Weight (kg)': data.groupby(column_name)['Weight'].mean().round(2),
        'Mean BMI': data.groupby(column_name)['BMI'].mean().round(2),
        'Mean Systolic': data.groupby(column_name)['Systolic'].mean().round(2),
        'Mean Diastolic': data.groupby(column_name)['Diastolic'].mean().round(2),
        'Normal uACR Counts': data[(data['uACR'] == 'Normal')].groupby([column_name]).size(),
        'Abnormal uACR Counts': data[(data['uACR'] == 'Abnormal')].groupby([column_name]).size(),
        'High Abnormal uACR Counts': data[(data['uACR'] == 'High Abnormal')].groupby([column_name]).size(),
        }
    table = pd.DataFrame(table).set_index(column_name).fillna(0)
    table = table.astype({'Normal uACR Counts': 'int', 'Abnormal uACR Counts': 'int', 'High Abnormal uACR Counts': 'int'})
    return table

In [11]:
eth_data = group_stats('Ethnicity')
eth_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
Ethnicity,,,,,,,,,
Any other,53.28,170.88,97.35,33.10,133.25,82.25,3,1,0
Asian other,50.09,168.34,84.01,29.66,128.96,78.87,15,7,1
Bangladeshi,59.20,167.91,80.63,28.44,152.71,84.71,6,1,0
Black African (Central Africa),51.42,168.65,83.70,29.49,138.00,83.93,10,5,0
Black African (East Africa),47.48,163.08,78.08,29.44,128.24,79.24,10,9,2
Black African (North Africa),54.88,170.75,77.33,26.58,130.00,80.83,5,1,0
Black African (South Africa),46.72,164.92,82.48,30.28,126.00,78.50,1,5,0
Black African (West Africa),50.98,166.03,81.88,29.79,136.51,82.84,264,198,37
Black African (unspecified),51.32,164.79,81.33,30.13,136.57,83.36,77,62,10


In [12]:
s_eth_data = group_stats('S_Ethnicity')
s_eth_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
S_Ethnicity,,,,,,,,,
Asian other,50.09,168.34,84.01,29.66,128.96,78.87,15,7,1
Black,50.88,165.83,81.55,29.75,136.17,82.66,430,335,52
Mixed,56.42,163.27,82.89,31.05,140.85,82.67,18,12,3
Other,53.28,170.88,97.35,33.10,133.25,82.25,3,1,0
South Asian,52.01,168.48,82.23,29.08,141.31,84.91,25,7,0
White,47.23,164.03,82.64,30.78,134.21,83.06,39,11,2


In [13]:
eth_black_data = group_stats('Ethnicity_Black')
eth_black_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
Ethnicity_Black,,,,,,,,,
False,49.79,165.99,83.38,30.31,135.52,82.78,87,28,3
True,51.06,165.79,81.58,29.78,136.31,82.65,443,345,55


In [14]:
gender_data = group_stats('Gender')
gender_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
Gender,,,,,,,,,
Female,51.20,165.57,81.31,29.74,136.80,82.57,276,237,27
Male,50.52,166.12,82.43,29.97,135.46,82.80,254,136,31


In [15]:
age_data = group_stats('Age_Category').iloc[[3,0,1,2,4], :]
age_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
Age_Category,,,,,,,,,
<25,21.80,168.81,74.43,26.11,126.37,75.19,37,15,0
25-40,33.32,167.09,79.95,28.70,127.67,80.11,100,45,12
41-55,48.45,166.07,84.44,30.67,136.43,83.81,198,141,15
56-70,61.02,165.06,81.79,30.15,139.54,84.20,162,143,22
>70,76.71,162.97,78.17,29.56,145.89,81.03,33,29,9


In [16]:
bp_cat_data = group_stats('BP_Category').iloc[[3,4,0,1,2], :]
bp_cat_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
BP_Category,,,,,,,,,
NORMAL,48.26,165.70,79.84,29.16,123.15,75.95,318,197,25
PRE HPT,53.67,165.82,83.43,30.49,144.04,88.72,76,70,10
HPT 1,53.57,165.79,84.34,30.81,151.25,90.05,110,73,14
HPT 2,58.33,167.53,86.83,30.94,174.63,98.81,23,28,8
HPT CRISIS,54.60,161.84,82.59,31.43,202.78,113.67,3,5,1


In [17]:
fam_data = group_stats('Family_KD')
fam_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
Family_KD,,,,,,,,,
Definitely not,50.55,165.40,81.09,29.72,134.60,82.09,263,219,29
Definitely yes,50.80,166.41,80.62,29.09,135.81,81.42,22,11,3
Not sure,51.35,166.27,82.78,30.06,138.23,83.50,245,143,26


In [18]:
bmi_data = group_stats('BMI_Category').iloc[[-1,0,1,2], :]
bmi_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Normal uACR Counts,Abnormal uACR Counts,High Abnormal uACR Counts
BMI_Category,,,,,,,,,
OVERWEIGHT,51.12,166.88,76.98,27.57,135.69,82.35,206,139,16
HEALTHY,46.97,169.20,66.08,23.01,129.62,78.53,117,58,9
OBESE,52.45,163.39,92.94,34.83,139.57,84.78,207,176,33
OVERWEIGHT,51.12,166.88,76.98,27.57,135.69,82.35,206,139,16


In [19]:
hist_features = ['Age', 'Height', 'Weight', 'BMI', 'Systolic', 'Diastolic']

# Initialize figure
fig = go.Figure()

# Add Traces with visibility settings
for idx, feature in enumerate(hist_features):
    fig.add_trace(go.Histogram(name=feature, x=data[feature], visible=(idx == 0)))

# Overlay histograms
fig.update_layout(barmode='overlay')

fig.update_layout(
    updatemenus=[
        dict(
            direction="down",
            showactive=True,
            x=-0.2,
            xanchor='left',
            y=0.9,
            yanchor='top',
            buttons=list([
                dict(label=feature,
                     method="update",
                     args=[{"visible": [feature == j for j in hist_features]},
                          {"xaxis.title": feature}])
                for feature in hist_features
            ]),
        )
    ],
    xaxis_title=hist_features[0], # Default x-axis title for the first histogram
    yaxis_title='Count', # y-axis title
    bargap=0.1
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Feature:", showarrow=False,
        x = -0.2, xref="paper", y=1, yref="paper", align="left")
    ]
)

# Set plot size
fig.update_layout(
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20),
)

fig.show()

In [20]:
# Variables
x_vars = ['uACR', 'BP_Category', 'S_Ethnicity']
y_var = 'Age'
k_var = ['Family_KD', 'BMI_Category', 'BP_Category', 'S_Ethnicity', 'Ethnicity_Black', 'Gender']

# Initial x_var
current_x_var = x_vars[0]

fig = go.Figure()

# Add traces for each combination of k_var values
for a in k_var:
    for k in data[a].unique():
        fig.add_trace(
            go.Bar(
                name=f'{k}',
                x=data[current_x_var].unique(),
                y=[data[y_var][(data[current_x_var] == x) & (data[a] == k)].mean() for x in data[current_x_var].unique()],
                visible=a == k_var[0]
            )
        )

# Create buttons for each k_var
buttons_k_var = []
for a in k_var:
    buttons_k_var.append(dict(
        method='update',
        label=a,
        args=[{
            'visible': [a == current for current in k_var for _ in data[current].unique()],
            'title.text': f'Mean Age by {a} and {current_x_var}'
        }]
    ))

# Create buttons for each x_var
buttons_x_var = []
for x in x_vars:
    buttons_x_var.append(dict(
        method='update',
        label=x,
        args=[{
            'x': [data[x].unique()] * len(k_var) * data[k_var[0]].nunique(),
            'y': [[data[y_var][(data[x] == val) & (data[a] == k)].mean() for val in data[x].unique()] for a in k_var for k in data[a].unique()],
            'title.text': f'Mean Age by {k_var[0]} and {x}'
        }]
    ))

# Update layout with dropdowns
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons_x_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.9,
            yanchor='top'
        ),
        dict(
            buttons=buttons_k_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.6,
            yanchor='top'
        )
    ],
    barmode='group',
    title=dict(text='Bivariate Analysis (Age)', x=0.01),
    xaxis_title=current_x_var,
    yaxis_title='Mean Age',
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20)
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Select x Variable:", showarrow=False,
             x=-0.3, xref="paper", y=1, yref="paper", align="left"),
        dict(text="Select Feature:", showarrow=False,
             x=-0.3, xref="paper", y=0.7, yref="paper", align="left")
    ]
)

fig.show()


In [21]:
# List of dataframes
dfs = [bp_cat_data, age_data, gender_data, fam_data, s_eth_data]
df_names = ['BP Categories', 'Age Ranges', 'Genders', 'Family KD', 'Ethnicities']

# Default dataframe
df = dfs[0]

# Variables
x_var = df.index
y_var = df[['Mean Age', 'Mean Height (cm)', 'Mean Weight (kg)', 'Mean BMI', 'Mean Systolic', 'Mean Diastolic']]

# Create the initial figure
plot = go.Figure()

# Add traces for each y_var
for i in y_var.columns:
    plot.add_trace(
        go.Bar(name=i, x=x_var, y=y_var[i], visible=i == 'Mean Age')
    )

# Add dropdown menus
plot.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(label=df_name,
                     method='update',
                     args=[{'x': [dfs[df_names.index(df_name)].index] * len(y_var.columns),
                            'y': [dfs[df_names.index(df_name)][col] for col in y_var.columns],
                            'visible': [i == y_var.columns[0] for i in y_var.columns]},
                           {'title': f'{y_var.columns[0]} by {df_name}'}])
                for df_name in df_names
            ]),
            direction='down',
            showactive=True,
            x=-0.2,
            xanchor='left',
            y=0.9,
            yanchor='top'
        ),
        dict(
            buttons=list([
                dict(label=i,
                     method='update',
                     args=[{'visible': [i == j for j in y_var.columns]},
                           {'title': f'{i} by {df_names[0]}'}])
                for i in y_var.columns
            ]),
            direction='down',
            showactive=True,
            x=-0.2,
            xanchor='left',
            y=0.6,
            yanchor='top'
        )
    ]
)

# Add annotation
plot.update_layout(
    annotations=[
        dict(text="Select Data:", showarrow=False,
             x=-0.2, xref="paper", y=1, yref="paper", align="left"),
        dict(text="Select Feature:", showarrow=False,
             x=-0.2, xref="paper", y=0.7, yref="paper", align="left")
    ],
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20),
    title=dict(text='Univariate Analysis', x=0.01)
)

plot.show()

In [32]:
# create suplots
fig = make_subplots(rows=1, cols=2, subplot_titles=('Systolic vs Age', 'Diastolic vs Age'))

# systolic scatter plot
fig.add_trace(
    go.Scatter(
        x=data['Age'], 
        y=data['Systolic'],
        mode='markers',
        name='Systolic'),
    row=1, col=1
)

# diastolic scatter plot
fig.add_trace(
    go.Scatter(
        x=data['Age'],
        y=data['Diastolic'],
        mode='markers',
        name='Diastolic'),
    row=1, col=2
)

# update x-axis and y-axis titles
fig.update_xaxes(title_text='Age', row=1, col=1)
fig.update_yaxes(title_text='Systolic (mmHg)', row=1, col=1)
fig.update_xaxes(title_text='Age', row=1, col=2)
fig.update_yaxes(title_text='Diastolic (mmHg)', row=1, col=2)

fig.show()

In [33]:
# Variables
marker_categories = ['Gender', 'BP_Category', 'BMI_Category', 'uACR']
current_marker = marker_categories[0]

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=('Systolic vs Age', 'Diastolic vs Age'))

# Define color mappings
color_mappings = {
    'Gender': {'Male': 'blue', 'Female': 'pink'},
    'BP_Category': {'NORMAL': 'blue', 'PRE HPT': 'green', 'HPT 1': 'yellow', 'HPT 2': 'orange', 'HPT CRISIS': 'red'},
    'BMI_Category': {'UNDERWEIGHT': 'blue', 'HEALTHY': 'green', 'OVERWEIGHT': 'orange', 'OBESE': 'red'},
    'uACR': {'Normal': 'yellow', 'Abnormal': 'orange', 'High Abnormal': 'red'}
}

# Add traces for initial marker category
for marker_cat in marker_categories:
    fig.add_trace(
        go.Scatter(
            x=data['Age'],
            y=data['Systolic'],
            mode='markers',
            marker=dict(color=[color_mappings[marker_cat][val] for val in data[marker_cat]]),
            name=f'Systolic ({marker_cat})',
            visible=marker_cat == current_marker
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=data['Age'],
            y=data['Diastolic'],
            mode='markers',
            marker=dict(color=[color_mappings[marker_cat][val] for val in data[marker_cat]]),
            name=f'Diastolic ({marker_cat})',
            visible=marker_cat == current_marker
        ),
        row=1, col=2
    )

# Create buttons for marker categories
buttons = []
for marker_cat in marker_categories:
    buttons.append(dict(
        method='update',
        label=marker_cat,
        args=[{
            'visible': [marker_cat == current for current in marker_categories for _ in range(2)],
            'title.text': [f'Systolic vs Age ({marker_cat})', f'Diastolic vs Age ({marker_cat})']
        }]
    ))

# Update layout with dropdown
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.5,
            xanchor='center',
            y=1.2,
            yanchor='top'
        )
    ],
    xaxis1_title='Age',
    yaxis1_title='Systolic Blood Pressure',
    xaxis2_title='Age',
    yaxis2_title='Diastolic Blood Pressure',
    autosize=False,
    width=1100,
    height=450,
    margin=dict(l=20, r=20, t=50, b=20)
)

# Show the figure
fig.show()